# Testing Notebook

In [1]:
import config as cfg
import os

import torch
from torch import nn
from torch.utils.data import DataLoader

from Data.data_loading import load_and_preprocess_data, create_tensor_from_dataframe, create_sequences, create_dataloaders 
from Training.train_matt import Trainer
from Training.basicEval import plotLoss, plotAccuracy, reportFinalMetrics
from Model.model_split import FrameTransformer, print_model_info

W0505 10:42:55.589000 12484 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
root_dir = os.getcwd()  # Use current working directory as root
data_dir = os.path.join(root_dir, 'Data')
csv_dir = os.path.join(data_dir, 'csv')
csv_file = os.path.join(csv_dir, 'trimmed_IMG_4097_detections.csv')

print("Data directory: ", data_dir)
print("CSV directory: ", csv_dir)
print("CSV file: ", csv_file)


model_dir = os.path.join(root_dir, 'Model')
save_model_dir = os.path.join(model_dir, 'Saved_Model')
print("Model directory: ", model_dir)
print("Saved model directory: ", save_model_dir)


Data directory:  c:\Users\flyer\OneDrive\Documents\Github\Github-Deep-Learning-Project\Data
CSV directory:  c:\Users\flyer\OneDrive\Documents\Github\Github-Deep-Learning-Project\Data\csv
CSV file:  c:\Users\flyer\OneDrive\Documents\Github\Github-Deep-Learning-Project\Data\csv\trimmed_IMG_4097_detections.csv
Model directory:  c:\Users\flyer\OneDrive\Documents\Github\Github-Deep-Learning-Project\Model
Saved model directory:  c:\Users\flyer\OneDrive\Documents\Github\Github-Deep-Learning-Project\Model\Saved_Model


In [3]:
df, transformer_max_ids_per_frame, frame_scaler = load_and_preprocess_data(csv_folder=csv_dir)

# 2. Create tensor from dataframe
all_data_tensor = create_tensor_from_dataframe(df, transformer_max_ids_per_frame)

# 3. Create input-output sequences
X, Y = create_sequences(all_data_tensor)

# 4. Create dataloaders for training and testing
train_loader, test_loader, train_prefetcher, test_prefetcher = create_dataloaders(X, Y)


All CSVs now have 599 frames after trimming
Minimum records per ID: 3
Average records per ID: 1763.55
Maximum records per ID: 3209

Minimum IDs (Vehicles) per frame: 5
Average IDs (Vehicles) per frame: 14.34
Maximum IDs (Vehicles) per frame: 20

After normalization:
X range: 0.0000 to 5.0000
Y range: 0.0000 to 5.0000
Height range: 0.0000 to 5.0000
Width range: 0.0000 to 5.0000
Frame range: 0.0000 to 5.0000
All data tensor shape: torch.Size([6, 599, 20, 5])


In [ ]:
# Adjust the hidden_size and sequence_length to match the input tensor dimensions
model = FrameTransformer(
        input_feature_size=cfg.NUM_INPUT_FEATURES, 
        num_ids=transformer_max_ids_per_frame, 
        sequence_length=X.size(1),  
        prediction_length=cfg.PREDICTION_LENGTH,
        hidden_size=64,  
        num_heads=cfg.NUM_HEADS,
        dropout_rate=cfg.DROPOUT_RATE
)

# Print the model summary
print_model_info(model, X) 

RuntimeError: Expected one of cpu, cuda, ipu, xpu, mkldnn, opengl, opencl, ideep, hip, ve, fpga, maia, xla, lazy, vulkan, mps, meta, hpu, mtia, privateuseone device type at start of device string: Cuda

In [ ]:
trainScript = Trainer(model, train_loader, test_loader)

trainScript.earlyStop(enable=True, patience=30, delta=0.01)
train_losses, val_losses, train_accs, val_accs, epoch_times = trainScript.train(
    num_epochs=cfg.EPOCHS, 
    learningRate=cfg.LEARNING_RATE, 
    criterion=nn.MSELoss(), 
    optimizer=torch.optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE)
)

trainScript.save_model(model, save_model_dir)

In [ ]:
plotLoss(train_losses, val_losses)
plotAccuracy(train_accs, val_accs)
reportFinalMetrics(train_losses, val_losses, train_accs, val_accs, epoch_times)